# 🛰️ Z3r0 — Serve (Colab as an authenticated OpenAI-compatible server)

Turns a Colab **T4** into a public endpoint the local `tui/` client (or any HTTP client) can call.
One Ollama server holds all three tags; **Ollama hot-swaps** the requested model on demand, so a
single URL serves `control | huihui | obliteratus` by `model` name. A small **FastAPI proxy** gates
access with a **Bearer API key** and maps the friendly keys → real Ollama tags.

```
client ──Bearer KEY──▶ cloudflared ──▶ FastAPI proxy ──▶ ollama /v1   (401 if key is wrong)
                                         model: control|huihui|obliteratus  → real tag
```

> ⚖️ The endpoint serves abliterated models — the API key is the **only** gate on a world-reachable
> URL. Use a strong key, never commit it, and stop the tunnel when done.

## 1 · Check GPU (Runtime → Change runtime type → **T4 GPU**)

In [ ]:
!nvidia-smi

## 2 · Config

In [ ]:
#@title 🔧 Serve config { display-mode: "form" }
import secrets
API_KEY = ""        #@param {type:"string"}
#@markdown _Leave API_KEY blank to auto-generate a strong one._
MAX_CONTEXT = 4096  #@param {type:"integer"}
PROXY_PORT = 9000   #@param {type:"integer"}

if not API_KEY.strip():
    API_KEY = secrets.token_urlsafe(24)

# Which models to expose (each is pulled once; ollama hot-swaps per request).
TAGS_TO_PULL = ["control", "huihui", "obliteratus"]

print("API_KEY:", API_KEY)
print("expose :", TAGS_TO_PULL, "| proxy port:", PROXY_PORT, "| ctx:", MAX_CONTEXT)

## 3 · Install (Ollama + proxy deps)\nCurrent Ollama installer needs `zstd` on Colab — install it first.

In [ ]:
import subprocess, sys
def sh(cmd):
    print("+", cmd); subprocess.run(cmd, shell=True, check=False)
sh("apt-get -qq install -y zstd")                      # ollama installer needs zstd to extract
sh("curl -fsSL https://ollama.com/install.sh | sh")
sh(f"{sys.executable} -m pip install -q fastapi 'uvicorn[standard]' httpx requests")
print("\u2705 install done")

## 4 · Registry + start Ollama + pull the tags
Reuses the Z3r0 registry/fallbacks. Builds `MODEL_MAP` = friendly key → the tag that actually pulled.

In [ ]:
import os, time, subprocess, requests

REGISTRY = {
    "control":     {"ollama": "deepseek-r1:8b"},
    "huihui":      {"ollama": "huihui_ai/deepseek-r1-abliterated:8b"},
    "obliteratus": {"ollama": "OBLITERATUS/DeepSeek-R1-Distill-Llama-8B-OBLITERATED"},
}
OLLAMA_FALLBACKS = {
    "huihui": ["huihui_ai/deepseek-r1-abliterated:8b-llama-distill"],
    "obliteratus": ["sjo/deepseek-r1-8b-llama-distill-abliterated-q8_0",
                    "huihui_ai/deepseek-r1-abliterated:8b-llama-distill"],
}

env = os.environ.copy()
env["OLLAMA_HOST"] = "0.0.0.0:11434"
env["OLLAMA_ORIGINS"] = "*"
env["OLLAMA_CONTEXT_LENGTH"] = str(MAX_CONTEXT)   # honored by recent ollama

def ollama_up():
    try: return requests.get("http://127.0.0.1:11434/api/tags", timeout=3).status_code == 200
    except Exception: return False

if not ollama_up():
    subprocess.Popen(["ollama", "serve"], env=env, stdout=subprocess.DEVNULL, stderr=subprocess.STDOUT)
    for _ in range(60):
        if ollama_up(): break
        time.sleep(2)
print("ollama up:", ollama_up())

def pull_first_working(key):
    tags = [REGISTRY[key]["ollama"]] + OLLAMA_FALLBACKS.get(key, [])
    for tag in tags:
        try:
            print(f"  pulling {key} -> {tag} ...")
            subprocess.run(["ollama", "pull", tag], check=True, env=env)
            return tag
        except Exception as e:
            print(f"   failed {tag}: {e}")
    raise RuntimeError(f"could not pull any tag for {key}")

MODEL_MAP = {key: pull_first_working(key) for key in TAGS_TO_PULL}
print("\u2705 MODEL_MAP:", MODEL_MAP)

## 5 · FastAPI proxy — Bearer auth + model-key mapping + streaming passthrough
Runs uvicorn in a background thread (idempotent). Endpoints: `/v1/models`, `/v1/chat/completions`
(stream + non-stream), `/healthz` (open).

In [ ]:
import json, threading, time, requests, httpx, uvicorn
from fastapi import FastAPI, Request, Header, HTTPException
from fastapi.responses import StreamingResponse, JSONResponse

OLLAMA = "http://127.0.0.1:11434"
app = FastAPI(title="Z3r0 serve proxy")

def _auth(authorization):
    if authorization != f"Bearer {API_KEY}":
        raise HTTPException(status_code=401, detail="invalid or missing API key")

@app.get("/healthz")
def healthz():
    return {"ok": True, "models": list(MODEL_MAP)}

@app.get("/v1/models")
def models(authorization: str = Header(None)):
    _auth(authorization)
    return {"object": "list", "data": [{"id": k, "object": "model", "owned_by": "z3r0"} for k in MODEL_MAP]}

@app.post("/v1/chat/completions")
async def chat(req: Request, authorization: str = Header(None)):
    _auth(authorization)
    body = await req.json()
    body["model"] = MODEL_MAP.get(body.get("model"), body.get("model"))   # friendly -> real tag
    if body.get("stream"):
        async def gen():
            async with httpx.AsyncClient(timeout=None) as c:
                async with c.stream("POST", f"{OLLAMA}/v1/chat/completions", json=body) as r:
                    async for chunk in r.aiter_raw():
                        yield chunk
        return StreamingResponse(gen(), media_type="text/event-stream",
                                 headers={"Cache-Control": "no-cache", "X-Accel-Buffering": "no"})
    async with httpx.AsyncClient(timeout=None) as c:
        r = await c.post(f"{OLLAMA}/v1/chat/completions", json=body)
        return JSONResponse(r.json(), status_code=r.status_code)

def _proxy_healthy():
    try: return requests.get(f"http://127.0.0.1:{PROXY_PORT}/healthz", timeout=2).status_code == 200
    except Exception: return False

if not _proxy_healthy():
    threading.Thread(
        target=lambda: uvicorn.run(app, host="0.0.0.0", port=PROXY_PORT, log_level="warning"),
        daemon=True).start()
    for _ in range(30):
        if _proxy_healthy(): break
        time.sleep(1)
print("\u2705 proxy healthy:", _proxy_healthy())

## 6 · Public URL (cloudflared) — forwards the **proxy** port

In [ ]:
import os, re, subprocess, time
if not os.path.exists("cloudflared"):
    subprocess.run("wget -q https://github.com/cloudflare/cloudflared/releases/latest/"
                   "download/cloudflared-linux-amd64 -O cloudflared && chmod +x cloudflared",
                   shell=True, check=True)
proc = subprocess.Popen(["./cloudflared", "tunnel", "--url", f"http://localhost:{PROXY_PORT}",
                         "--no-autoupdate"], stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                        text=True, bufsize=1)
PUBLIC_URL = None; t0 = time.time()
while time.time() - t0 < 60:
    ldine = proc.stdout.readline()
    if not line:
        if proc.poll() is not None: break
        continue
    m = re.search(r"https://[-a-z0-9.]+\.trycloudflare\.com", line)
    if m:
        PUBLIC_URL = m.group(0); break
globals()["_TUNNEL_PROC"] = proc
print("\U0001f310 PUBLIC URL:", PUBLIC_URL)

## 7 · Verify + the config to paste into the TUI

In [ ]:
import requests
# local proxy self-test (should be 401 without the key, 200 with it)
u = f"http://127.0.0.1:{PROXY_PORT}/v1/models"
print("local no-auth ->", requests.get(u).status_code, "(expect 401)")
print("local auth    ->", requests.get(u, headers={"Authorization": f"Bearer {API_KEY}"}).status_code, "(expect 200)")

print("\n" + "=" * 64)
print("PUBLIC URL :", PUBLIC_URL)
print("API KEY    :", API_KEY)
print("MODELS     :", list(MODEL_MAP))
print("=" * 64)

print("\n# tui/z3r0.config.json  (or export these as env vars):")
print(json.dumps({"baseUrl": PUBLIC_URL, "apiKey": API_KEY, "model": "control"}, indent=2))

print("\n# curl smoke test (streaming) from anywhere:")
print('curl -N -H "Authorization: Bearer ' + API_KEY + '" \\')
print('  -H "Content-Type: application/json" \\')
print('  -d \'{"model":"control","stream":true,"messages":[{"role":"user","content":"hi in 3 words"}]}\' \\')
print("  " + str(PUBLIC_URL) + "/v1/chat/completions")

## Notes
- The cloudflared URL is **ephemeral** — it changes every run and dies when the Colab runtime
  disconnects (~12h). Re-paste it into the TUI (`Ctrl+E`) when it rotates.
- First request to a not-yet-loaded model pays a few seconds of VRAM load (T4 holds one 8B; switching
  models evicts the previous one).
- Stop sharing by interrupting the cloudflared cell / disconnecting the runtime.